In [1]:
import pandas as pd

# File path
DATA_PATH = r"C:\Users\sayee\OneDrive\Desktop\emotion_predict\training_data.xlsx"

# Load dataset
df = pd.read_excel(DATA_PATH)

# Basic checks
print("Dataset Loaded Successfully!\n")

print("Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns)

print("\nFirst 5 Rows:")
print(df.head())

Dataset Loaded Successfully!

Shape:
(1200, 13)

Columns:
Index(['id', 'journal_text', 'ambience_type', 'duration_min', 'sleep_hours',
       'energy_level', 'stress_level', 'time_of_day', 'previous_day_mood',
       'face_emotion_hint', 'reflection_quality', 'emotional_state',
       'intensity'],
      dtype='object')

First 5 Rows:
   id                                       journal_text ambience_type  \
0   1  The ocean ambience helped me stop drifting and...         ocean   
1   2  I tried to relax during the forest ambience, y...        forest   
2   3  The forest session slowed my thoughts and I fe...        forest   
3   4  the mountain ambience was pleasant, though i c...      mountain   
4   5  The rain session gave me a pause, but the pres...          rain   

   duration_min  sleep_hours  energy_level  stress_level time_of_day  \
0            12          6.5             4             2   afternoon   
1            35          6.0             2             4     evening   
2 

In [2]:
print("Missing Values:\n")
print(df.isnull().sum())

Missing Values:

id                      0
journal_text            0
ambience_type           0
duration_min            0
sleep_hours             7
energy_level            0
stress_level            0
time_of_day             0
previous_day_mood      15
face_emotion_hint     123
reflection_quality      0
emotional_state         0
intensity               0
dtype: int64


In [3]:
# Fill numeric
df['sleep_hours'] = df['sleep_hours'].fillna(df['sleep_hours'].mean())

# Fill categorical
df['previous_day_mood'] = df['previous_day_mood'].fillna("unknown")
df['face_emotion_hint'] = df['face_emotion_hint'].fillna("unknown")

print("\nMissing values after handling:")
print(df.isnull().sum())



Missing values after handling:
id                    0
journal_text          0
ambience_type         0
duration_min          0
sleep_hours           0
energy_level          0
stress_level          0
time_of_day           0
previous_day_mood     0
face_emotion_hint     0
reflection_quality    0
emotional_state       0
intensity             0
dtype: int64


In [4]:
import re

def clean_text(text):
    text = str(text).lower()  # lowercase
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()  # remove extra spaces
    return text

df['clean_text'] = df['journal_text'].apply(clean_text)

print("\nSample cleaned text:")
print(df[['journal_text', 'clean_text']].head())


Sample cleaned text:
                                        journal_text  \
0  The ocean ambience helped me stop drifting and...   
1  I tried to relax during the forest ambience, y...   
2  The forest session slowed my thoughts and I fe...   
3  the mountain ambience was pleasant, though i c...   
4  The rain session gave me a pause, but the pres...   

                                          clean_text  
0  the ocean ambience helped me stop drifting and...  
1  i tried to relax during the forest ambience ye...  
2  the forest session slowed my thoughts and i fe...  
3  the mountain ambience was pleasant though i ca...  
4  the rain session gave me a pause but the press...  


In [5]:
from sklearn.preprocessing import LabelEncoder

le_dict = {}

categorical_cols = [
    'ambience_type',
    'time_of_day',
    'previous_day_mood',
    'face_emotion_hint',
    'reflection_quality',
    'emotional_state'
]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le  # store encoders for later use

print("\nEncoded columns sample:")
print(df.head())


Encoded columns sample:
   id                                       journal_text  ambience_type  \
0   1  The ocean ambience helped me stop drifting and...              3   
1   2  I tried to relax during the forest ambience, y...              1   
2   3  The forest session slowed my thoughts and I fe...              1   
3   4  the mountain ambience was pleasant, though i c...              2   
4   5  The rain session gave me a pause, but the pres...              4   

   duration_min  sleep_hours  energy_level  stress_level  time_of_day  \
0            12     6.500000             4             2            0   
1            35     6.000000             2             4            2   
2             3     5.989522             2             1            4   
3            25     7.000000             4             4            4   
4            25     5.000000             3             5            0   

   previous_day_mood  face_emotion_hint  reflection_quality  emotional_state  \
0    

In [6]:
print("\nFinal Dataset Info:")
print(df.info())


Final Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  1200 non-null   int64  
 1   journal_text        1200 non-null   object 
 2   ambience_type       1200 non-null   int64  
 3   duration_min        1200 non-null   int64  
 4   sleep_hours         1200 non-null   float64
 5   energy_level        1200 non-null   int64  
 6   stress_level        1200 non-null   int64  
 7   time_of_day         1200 non-null   int64  
 8   previous_day_mood   1200 non-null   int64  
 9   face_emotion_hint   1200 non-null   int64  
 10  reflection_quality  1200 non-null   int64  
 11  emotional_state     1200 non-null   int64  
 12  intensity           1200 non-null   int64  
 13  clean_text          1200 non-null   object 
dtypes: float64(1), int64(11), object(2)
memory usage: 131.4+ KB
None


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1,2)
)

X_text = tfidf.fit_transform(df['clean_text'])

print("Text feature shape:", X_text.shape)

Text feature shape: (1200, 1978)


In [8]:
meta_cols = [
    'ambience_type',
    'duration_min',
    'sleep_hours',
    'energy_level',
    'stress_level',
    'time_of_day',
    'previous_day_mood',
    'face_emotion_hint',
    'reflection_quality'
]

X_meta = df[meta_cols]

print("Metadata shape:", X_meta.shape)
print(X_meta.head())

Metadata shape: (1200, 9)
   ambience_type  duration_min  sleep_hours  energy_level  stress_level  \
0              3            12     6.500000             4             2   
1              1            35     6.000000             2             4   
2              1             3     5.989522             2             1   
3              2            25     7.000000             4             4   
4              4            25     5.000000             3             5   

   time_of_day  previous_day_mood  face_emotion_hint  reflection_quality  
0            0                  2                  0                   0  
1            2                  0                  5                   2  
2            4                  4                  1                   0  
3            4                  1                  0                   2  
4            0                  6                  4                   0  


In [9]:
from scipy.sparse import hstack, csr_matrix
import numpy as np

# Preview combine (unscaled) — the scaled version is built in the next cells
X_preview = hstack([X_text, csr_matrix(X_meta.values.astype(float))])

print("Preview feature shape (unscaled):", X_preview.shape)


Preview feature shape (unscaled): (1200, 1987)


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_meta_scaled = scaler.fit_transform(X_meta.values)  # .values → suppress feature-name warning

print("Scaled metadata sample:")
print(X_meta_scaled[:5])


Scaled metadata sample:
[[ 0.71169907 -0.50359676  0.34129202  0.71218859 -0.73284327 -1.44790372
  -0.3629037  -1.41807844 -1.2053351 ]
 [-0.73459878  2.49581426  0.00700517 -0.73633058  0.69477349 -0.11700792
  -1.52419553  1.16651167  1.22763735]
 [-0.73459878 -1.67727934  0.         -0.73633058 -1.44665166  1.21388787
   0.79838814 -0.90116042 -1.2053351 ]
 [-0.01144986  1.19172251  0.67557888  0.71218859  0.69477349  1.21388787
  -0.94354961 -1.41807844  1.22763735]
 [ 1.43484799  1.19172251 -0.66156853 -0.01207099  1.40858188 -1.44790372
   1.95967997  0.64959365 -1.2053351 ]]


In [11]:
from scipy.sparse import hstack

X_train = hstack([X_text, X_meta_scaled])

print("Final feature shape:", X_train.shape)

Final feature shape: (1200, 1987)


In [12]:
y_state_train = df['emotional_state']
y_intensity_train = df['intensity']

In [13]:
from sklearn.svm import LinearSVC

clf = LinearSVC(max_iter=7000)

clf.fit(X_train, df['emotional_state'])

print("SVM Model Trained ✅")

SVM Model Trained ✅


In [14]:
from sklearn.metrics import accuracy_score

y_pred = clf.predict(X_train)

acc = accuracy_score(df['emotional_state'], y_pred)

print("Training Accuracy:", acc)

Training Accuracy: 0.8383333333333334


In [15]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression()

reg.fit(X_train, df['intensity'])

print("Intensity model trained ✅")

Intensity model trained ✅


In [16]:
from sklearn.metrics import mean_squared_error

y_pred_intensity = reg.predict(X_train)

mse = mean_squared_error(df['intensity'], y_pred_intensity)

print("Training MSE:", mse)

Training MSE: 0.6265098249097197


In [17]:
# Predict emotional state
state_pred = clf.predict(X_train)

# Predict intensity
intensity_pred = reg.predict(X_train)

print("Sample Predictions:\n")

for i in range(5):
    print(f"Predicted State: {state_pred[i]}, Predicted Intensity: {round(intensity_pred[i],2)}")

Sample Predictions:

Predicted State: 1, Predicted Intensity: 3.0
Predicted State: 5, Predicted Intensity: 3.0
Predicted State: 0, Predicted Intensity: 3.0
Predicted State: 3, Predicted Intensity: 1.0
Predicted State: 4, Predicted Intensity: 4.53


In [18]:
# Get label encoder for emotional_state
le_state = le_dict['emotional_state']

# Convert numbers back to labels
state_pred_labels = le_state.inverse_transform(state_pred)

print("Sample Predictions (Readable):\n")

for i in range(5):
    print(f"State: {state_pred_labels[i]}, Intensity: {round(intensity_pred[i],2)}")

Sample Predictions (Readable):

State: focused, Intensity: 3.0
State: restless, Intensity: 3.0
State: calm, Intensity: 3.0
State: neutral, Intensity: 1.0
State: overwhelmed, Intensity: 4.53


In [19]:
print(le_dict['time_of_day'].classes_)

['afternoon' 'early_morning' 'evening' 'morning' 'night']


In [20]:
def decision_engine(state, intensity, stress, energy, time_of_day):

    # 🔴 1. HIGH DISTRESS
    if state == "overwhelmed":
        if intensity >= 4 or stress >= 4:
            if time_of_day in [2, 4]:  # evening=2, night=4
                return "box_breathing", "now"
            else:
                return "grounding", "now"
        else:
            return "journaling", "within_15_min"

    # 🟡 2. RESTLESS / AGITATED
    elif state == "restless":
        if energy >= 3:
            return "movement", "within_15_min"
        else:
            return "sound_therapy", "now"

    # 🟢 3. CALM STATE
    elif state == "calm":
        if energy >= 4:
            return "deep_work", "now"
        elif energy >= 2:
            return "light_planning", "later_today"
        else:
            return "rest", "later_today"

    # 🔵 4. FOCUSED STATE
    elif state == "focused":
        if energy >= 3:
            return "deep_work", "now"
        else:
            return "light_planning", "within_15_min"

    # ⚪ 5. NEUTRAL STATE
    elif state == "neutral":
        if energy >= 3:
            return "light_planning", "later_today"
        else:
            return "rest", "later_today"

    # 🟣 6. MIXED STATE
    elif state == "mixed":
        if stress >= 3:
            return "journaling", "within_15_min"
        else:
            return "pause", "now"

    # 🌙 7. TIME-BASED SLEEP NUDGE (reachable default path)
    if time_of_day in [2, 4] and energy <= 2:  # evening/night + low energy
        return "sleep", "tonight"

    # 🔚 Fallback
    return "pause", "later_today"


In [21]:
for i in range(5):
    action, timing = decision_engine(
        state_pred_labels[i],
        intensity_pred[i],
        df['stress_level'].iloc[i],
        df['energy_level'].iloc[i],
        df['time_of_day'].iloc[i]
    )
    
    print(f"{state_pred_labels[i]} -> {action} ({timing})")

focused -> deep_work (now)
restless -> sound_therapy (now)
calm -> light_planning (later_today)
neutral -> light_planning (later_today)
overwhelmed -> grounding (now)


In [22]:
decision_scores = clf.decision_function(X_train)

# Take highest score per row
confidence_scores = decision_scores.max(axis=1)

print("Raw confidence scores:")
print(confidence_scores[:5])

Raw confidence scores:
[0.9321567  0.84504386 0.7067669  0.66132859 0.95321014]


In [23]:
import numpy as np

# Normalize
conf_min = confidence_scores.min()
conf_max = confidence_scores.max()

confidence_normalized = (confidence_scores - conf_min) / (conf_max - conf_min)

print("Normalized confidence:")
print(confidence_normalized[:5])

Normalized confidence:
[0.80305791 0.75714927 0.68427703 0.66033094 0.81415312]


In [24]:
uncertain_flags = confidence_normalized < 0.5

print("Uncertainty flags:")
print(uncertain_flags[:5])

Uncertainty flags:
[False False False False False]


In [25]:
print("Min confidence:", confidence_normalized.min())
print("Max confidence:", confidence_normalized.max())

Min confidence: 0.0
Max confidence: 1.0


In [26]:
print("Number of uncertain predictions:", sum(uncertain_flags))

Number of uncertain predictions: 448


In [27]:
from scipy.sparse import hstack, csr_matrix

results = []

for i in range(len(df)):

    # -------- TEXT --------
    text = df["clean_text"].iloc[i]
    text_vec = tfidf.transform([text])          # fixed: was 'vectorizer' (undefined)

    # -------- METADATA --------
    meta_row = X_meta.iloc[i].values            # fixed: was 'metadata' (undefined)
    meta_scaled = scaler.transform([meta_row])

    # -------- COMBINE --------
    X_row = hstack([text_vec, csr_matrix(meta_scaled)])

    # -------- PREDICTIONS --------
    state_p = clf.predict(X_row)[0]
    intensity_p = reg.predict(X_row)[0]

    # -------- CONFIDENCE --------
    decision_score = clf.decision_function(X_row)
    conf_raw = decision_score.max()
    conf_norm = (conf_raw - conf_min) / (conf_max - conf_min)

    uncertain = conf_norm < 0.4

    # -------- DECISION ENGINE --------
    state_label = le_dict['emotional_state'].inverse_transform([state_p])[0]
    action, when = decision_engine(
        state_label,
        intensity_p,
        meta_row[4],  # stress_level
        meta_row[3],  # energy_level
        meta_row[5]   # time_of_day
    )

    # -------- STORE --------
    results.append({
        "text":       text[:60],
        "state":      state_label,
        "intensity":  round(float(intensity_p), 2),
        "action":     action,
        "when":       when,
        "confidence": round(float(conf_norm), 2),
        "uncertain":  bool(uncertain)
    })

results_df = pd.DataFrame(results)
print("DONE ✅")
print(results_df.head(10))


DONE ✅
                                                text        state  intensity  \
0  the ocean ambience helped me stop drifting and...      focused       3.00   
1  i tried to relax during the forest ambience ye...     restless       3.00   
2  the forest session slowed my thoughts and i fe...         calm       3.00   
3  the mountain ambience was pleasant though i ca...      neutral       1.00   
4  the rain session gave me a pause but the press...  overwhelmed       4.53   
5  after the forest track i feel peaceful and les...         calm       3.35   
6  nothing strong came up during the rain session...      neutral       1.68   
7  even with the mountain session my mind kept ju...     restless       3.96   
8  i couldnt really settle into the cafe track i ...     restless       4.00   
9  the mountain ambience helped me stop drifting ...      focused       2.52   

           action           when  confidence  uncertain  
0       deep_work            now        0.80      Fals

In [28]:
feature_names = tfidf.get_feature_names_out()

print("Sample text features:")
print(feature_names[:20])

Sample text features:
['able' 'able to' 'about' 'about emails' 'about otehr' 'about other'
 'about work' 'action' 'actually' 'actually able' 'actually helped'
 'affected' 'affected it' 'after' 'after after' 'after bit' 'after but'
 'after couldnt' 'after few' 'after it']


In [29]:
feature_names = tfidf.get_feature_names_out()

# Get coefficients
coef = clf.coef_

print("Shape of coef:", coef.shape)

Shape of coef: (6, 1987)


In [30]:
top_indices = np.argsort(coef[0])[-10:]

print("Top words for class 0:")
for i in top_indices:
    print(feature_names[i])

Top words for class 0:
felt lighter
lighter than
felt more
first okay
lighter
surprisingly okay
surprisingly
less
my breathing
reason still


In [31]:
print(le_dict['emotional_state'].classes_)

['calm' 'focused' 'mixed' 'neutral' 'overwhelmed' 'restless']


In [32]:
classes = le_dict['emotional_state'].classes_

for idx, label in enumerate(classes):
    print(f"\nTop words for {label}:")
    
    top_indices = np.argsort(coef[idx])[-10:]
    
    for i in top_indices:
        print(feature_names[i])


Top words for calm:
felt lighter
lighter than
felt more
first okay
lighter
surprisingly okay
surprisingly
less
my breathing
reason still

Top words for focused:
planning
started planning
concentrate
guess mind
guess felt
able
able to
clearer
organized
end got

Top words for mixed:
heavy after
half
changed got
time could
but still
end back
not
in between
but not
between

Top words for neutral:
okay overall
overall
of blank
blank
normal
shifted still
felt ordinary
ordinary
nothing
pretty even

Top words for overwhelmed:
than expected
relax much
couldnt relax
pretty overloaded
mentally flooded
everything
felt mentally
flooded
overloaded
drained

Top words for restless:
itchy
itchy in
restless
kept
guess got
restless even
reason helped
tasks
still mentally
felt distracted


In [33]:
# TEXT ONLY FEATURES
X_text_only = tfidf.transform(df["clean_text"])

# Train model
from sklearn.svm import LinearSVC

clf_text = LinearSVC(max_iter=2000)
clf_text.fit(X_text_only, y_state_train)

# Accuracy
acc_text = clf_text.score(X_text_only, y_state_train)

print("Text-only Accuracy:", acc_text)

Text-only Accuracy: 0.8191666666666667


## 📊 Ablation Study

An ablation study evaluated the contribution of metadata features.

- **Text-only model:** 0.819 accuracy
- **Text + metadata model:** 0.838 accuracy

Contextual features (stress level, energy level, time of day) provide complementary signal that improves prediction.

In [34]:
# Get predictions on training data
y_pred = clf.predict(X_train)

# Compare with actual
errors = df[y_pred != y_state_train]

print("Number of errors:", len(errors))

Number of errors: 194


In [35]:
# Step 1: Create mask
mask = y_pred != y_state_train

# Step 2: Filter dataframe
error_df = df[mask].copy()

# Step 3: Assign ONLY error predictions
error_df["predicted"] = y_pred[mask]

# Step 4: View
error_df[["journal_text", "emotional_state", "predicted"]].head(10)

,journal_text,emotional_state,predicted
484,I guess mind was all over the place.,3,1
485,kinda calm now,4,1
491,For some reason okay session.,3,0
501,At first kept thinking about work. Then it shi...,5,3
502,felt better after a bit ...,4,5
503,"Honestly felt lighter, not fully though.",0,1
505,By teh end still a bit off tbh.,2,5
506,kept thinking about work,0,3
507,At first felt good for a moment.,5,2
508,by the end honestly not much change.,1,5


In [36]:
import numpy as np
from scipy.sparse import hstack, csr_matrix

def final_pipeline(text, meta_row):
    """
    End-to-end prediction for a single input.
    Parameters: text (str), meta_row (array of 9 metadata values in meta_cols order)
    Returns: dict with state, intensity, action, when, confidence, uncertain
    """
    # 1. Text preprocessing
    cleaned   = clean_text(str(text).strip())
    augmented = cleaned if len(cleaned.split()) > 2 else cleaned + " neutral state feeling"
    text_vec  = tfidf.transform([augmented])

    # 2. Combine features
    meta_scaled = scaler.transform([meta_row])
    X_row       = hstack([text_vec, csr_matrix(meta_scaled)])

    # 3. Predict emotional state
    state_enc   = clf.predict(X_row)[0]
    state_label = le_dict["emotional_state"].inverse_transform([state_enc])[0]

    # 4. Predict intensity — clip to valid scale [1, 5]
    intensity = float(np.clip(reg.predict(X_row)[0], 1.0, 5.0))

    # 5. Confidence — sigmoid normalisation using training conf_min / conf_max
    #    FIX: replaced undefined `conf_raw_history` with training statistics
    conf_raw  = float(np.max(clf.decision_function(X_row)))
    centre    = (conf_min + conf_max) / 2.0
    scale     = max(conf_max - conf_min, 1e-6)
    conf_norm = float(np.clip(1 / (1 + np.exp(-4 * (conf_raw - centre) / scale)), 0.0, 1.0))
    uncertain = conf_norm < 0.50

    # 6. Short / ambiguous input override
    if len(cleaned.split()) <= 2 or cleaned in {"ok", "fine", "good", "normal", ""}:
        if state_label not in {"neutral", "calm"}:
            state_label = "neutral"
        intensity  = float(np.clip(intensity, 2.0, 3.5))
        conf_norm  = min(conf_norm, 0.45)
        uncertain  = True

    # 7. Decision logic
    action, when = decision_engine(
        state_label,
        intensity,
        meta_row[4],   # stress_level
        meta_row[3],   # energy_level
        meta_row[5],   # time_of_day
    )

    return {
        "state":      state_label,
        "intensity":  round(intensity, 2),
        "action":     action,
        "when":       when,
        "confidence": round(conf_norm, 3),
        "uncertain":  bool(uncertain),
    }


## 🧪 Short-Input Robustness

Tested on very short inputs (*"ok"*, *"fine"*). The model produced predictions with low confidence scores (e.g., 0.06, 0.18) and correctly flagged them as uncertain, preventing overconfident outputs.

---
## 🧪 Evaluation on Test Dataset

The model was trained on `training_data.xlsx`. The cells below apply the **same pipeline** to `test_data.xlsx` and evaluate real generalisation performance.

In [37]:
# ── Load test dataset ──────────────────────────────────────────────────
import pandas as pd

TEST_PATH = r"C:\Users\sayee\OneDrive\Desktop\emotion_predict\testing_data.xlsx"

df_test = pd.read_excel(TEST_PATH)

print('Test dataset loaded ✅')
print('Shape  :', df_test.shape)
print('Columns:', list(df_test.columns))
df_test.head()


Test dataset loaded ✅
Shape  : (120, 11)
Columns: ['id', 'journal_text', 'ambience_type', 'duration_min', 'sleep_hours', 'energy_level', 'stress_level', 'time_of_day', 'previous_day_mood', 'face_emotion_hint', 'reflection_quality']


,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality
0,10001,woke up feeling more organized mentally. i was...,cafe,4,8.5,3,1,night,mixed,happy_face,vague
1,10002,started off distracted most of the time. this ...,mountain,4,8.5,1,2,afternoon,mixed,happy_face,clear
2,10003,kinda calm ...,cafe,15,8.5,2,5,evening,calm,happy_face,vague
3,10004,after the session i felt able to think straigh...,ocean,7,7.0,2,3,morning,overwhelmed,none,clear
4,10005,lowkey felt pretty grounded. i had to restart ...,ocean,20,8.5,1,5,afternoon,calm,tired_face,vague


In [38]:
# ── Preprocess test data ───────────────────────────────────────────────

# Step 1: Missing value imputation
# scaler.mean_[2] == training mean of sleep_hours — no need to reference df
train_sleep_mean = scaler.mean_[2]   # index 2 = sleep_hours in meta_cols
df_test['sleep_hours']       = df_test['sleep_hours'].fillna(train_sleep_mean)
df_test['previous_day_mood'] = df_test['previous_day_mood'].fillna('unknown')
df_test['face_emotion_hint'] = df_test['face_emotion_hint'].fillna('unknown')

# Step 2: Clean text
df_test['clean_text'] = df_test['journal_text'].apply(clean_text)

# Step 3: Encode only columns that exist in the test file
# 'emotional_state' is what we PREDICT — skip it if absent
for col in categorical_cols:
    if col not in df_test.columns:
        continue
    known_classes = set(le_dict[col].classes_)
    df_test[col] = df_test[col].astype(str).apply(
        lambda x, kc=known_classes, le=le_dict[col]: x if x in kc else le.classes_[0]
    )
    df_test[col] = le_dict[col].transform(df_test[col])

# Flag whether ground-truth labels are present
has_labels = ('emotional_state' in df_test.columns) and ('intensity' in df_test.columns)

print('Test data preprocessed ✅')
print(f'Ground-truth labels present: {has_labels}')
print('Missing values after handling:')
print(df_test.isnull().sum())


Test data preprocessed ✅
Ground-truth labels present: False
Missing values after handling:
id                    0
journal_text          0
ambience_type         0
duration_min          0
sleep_hours           0
energy_level          0
stress_level          0
time_of_day           0
previous_day_mood     0
face_emotion_hint     0
reflection_quality    0
clean_text            0
dtype: int64


In [39]:
# ── Build test feature matrix ──────────────────────────────────────────
from scipy.sparse import hstack, csr_matrix

# Text: transform() only — vocabulary fixed from training
X_test_text = tfidf.transform(df_test['clean_text'])

# Metadata: transform() only — scaler fitted on training
X_meta_test        = df_test[meta_cols]
X_meta_test_scaled = scaler.transform(X_meta_test.values)

X_test = hstack([X_test_text, X_meta_test_scaled])

print('Test feature matrix shape :', X_test.shape)
print('Train feature matrix shape:', X_train.shape)


Test feature matrix shape : (120, 1987)
Train feature matrix shape: (1200, 1987)


In [40]:
# ── Emotional State — Predict & Evaluate ───────────────────────────────
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

y_pred_test_state      = clf.predict(X_test)
state_pred_test_labels = le_dict['emotional_state'].inverse_transform(y_pred_test_state)

print('Sample predicted states:')
for i in range(min(5, len(df_test))):
    print(f'  [{i}] {state_pred_test_labels[i]}')

if has_labels:
    y_test_state = df_test['emotional_state']
    train_acc    = accuracy_score(df['emotional_state'], clf.predict(X_train))
    test_acc     = accuracy_score(y_test_state, y_pred_test_state)
    print(f'\nTraining Accuracy : {train_acc:.4f}')
    print(f'Test Accuracy     : {test_acc:.4f}')
    print()
    print('Classification Report (Test):')
    print(classification_report(
        y_test_state,
        y_pred_test_state,
        target_names=le_dict['emotional_state'].classes_
    ))
else:
    print('\n(No ground-truth labels — accuracy cannot be computed)')


Sample predicted states:
  [0] focused
  [1] restless
  [2] focused
  [3] focused
  [4] calm

(No ground-truth labels — accuracy cannot be computed)


In [41]:
# ── Confusion Matrix (Test) ─────────────────────────────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

if has_labels:
    class_names = le_dict['emotional_state'].classes_
    cm = confusion_matrix(y_test_state, y_pred_test_state)

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=35, ha='right')
    ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Confusion Matrix — Emotional State (Test Set)')
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print('Confusion matrix skipped — no ground-truth labels in test file.')


Confusion matrix skipped — no ground-truth labels in test file.


In [42]:
# ── Intensity — Predict & Evaluate ─────────────────────────────────────
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred_test_intensity = reg.predict(X_test)

print('Sample predicted intensities:')
for i in range(min(5, len(df_test))):
    print(f'  [{i}] {round(float(y_pred_test_intensity[i]), 2)}')

if has_labels:
    y_test_intensity = df_test['intensity']
    test_mse  = mean_squared_error(y_test_intensity, y_pred_test_intensity)
    test_rmse = np.sqrt(test_mse)
    test_mae  = mean_absolute_error(y_test_intensity, y_pred_test_intensity)
    test_r2   = r2_score(y_test_intensity, y_pred_test_intensity)
    train_mse = mean_squared_error(df['intensity'], reg.predict(X_train))
    print(f'\nTraining MSE : {train_mse:.4f}')
    print(f'Test MSE     : {test_mse:.4f}')
    print(f'Test RMSE    : {test_rmse:.4f}')
    print(f'Test MAE     : {test_mae:.4f}')
    print(f'Test R2      : {test_r2:.4f}')
else:
    print('\n(No ground-truth intensity — MSE cannot be computed)')


Sample predicted intensities:
  [0] 0.7
  [1] 4.42
  [2] 4.37
  [3] 0.52
  [4] 2.0

(No ground-truth intensity — MSE cannot be computed)


In [43]:
# ── Decision Engine — Test Samples ─────────────────────────────────────

print(f"{'State':<14} {'Intensity':>9}   {'Action':<18} {'When'}")
print('-' * 60)

for i in range(min(10, len(df_test))):
    action, timing = decision_engine(
        state_pred_test_labels[i],
        float(np.clip(y_pred_test_intensity[i], 1.0, 5.0)),
        df_test['stress_level'].iloc[i],
        df_test['energy_level'].iloc[i],
        df_test['time_of_day'].iloc[i],
    )
    print(f"{state_pred_test_labels[i]:<14} "
          f"{round(float(y_pred_test_intensity[i]), 2):>9}   "
          f"{action:<18} {timing}")


State          Intensity   Action             When
------------------------------------------------------------
focused              0.7   deep_work          now
restless            4.42   sound_therapy      now
focused             4.37   light_planning     within_15_min
focused             0.52   light_planning     within_15_min
calm                 2.0   rest               later_today
mixed               7.82   pause              now
restless             6.7   sound_therapy      now
overwhelmed         0.06   grounding          now
restless           -8.47   movement           within_15_min
overwhelmed         8.33   box_breathing      now


In [44]:
# ── Error Analysis — Test Set ───────────────────────────────────────────

if has_labels:
    test_mask     = y_pred_test_state != df_test['emotional_state'].values
    test_error_df = df_test[test_mask].copy()
    test_error_df['predicted'] = le_dict['emotional_state'].inverse_transform(
        y_pred_test_state[test_mask]
    )
    test_error_df['actual'] = le_dict['emotional_state'].inverse_transform(
        df_test['emotional_state'].values[test_mask]
    )
    print(f"Test errors: {test_mask.sum()} / {len(df_test)} "
          f"({test_mask.mean()*100:.1f}%)")
    print()
    print(test_error_df[['journal_text', 'actual', 'predicted']].head(10).to_string())
else:
    print('Error analysis skipped — no ground-truth labels in test file.')


Error analysis skipped — no ground-truth labels in test file.


In [45]:
results_df.head()

,text,state,intensity,action,when,confidence,uncertain
0,the ocean ambience helped me stop drifting and...,focused,3.00,deep_work,now,0.80,False
1,i tried to relax during the forest ambience ye...,restless,3.00,sound_therapy,now,0.76,False
2,the forest session slowed my thoughts and i fe...,calm,3.00,light_planning,later_today,0.68,False
3,the mountain ambience was pleasant though i ca...,neutral,1.00,light_planning,later_today,0.66,False
4,the rain session gave me a pause but the press...,overwhelmed,4.53,grounding,now,0.81,False


In [46]:
y_state_pred = clf.predict(X_test)
y_intensity_pred = reg.predict(X_test)
print(conf_norm.shape)
print(type(conf_norm))
print(conf_norm.shape)

()
<class 'numpy.float64'>
()


In [48]:
# ────────────────────────────────────────────────────────────────
# FINAL OUTPUT: Generate predictions + supportive messages
# ────────────────────────────────────────────────────────────────

import pandas as pd

final_output = []

for i in range(len(df_test)):
    text = df_test['journal_text'].iloc[i]
    meta = X_meta_test.iloc[i].values
    
    result = final_pipeline(text, meta)
    
    final_output.append({
        "id": int(df_test["id"].iloc[i]),
        "predicted_state": result['state'],
        "predicted_intensity": round(float(result['intensity']), 2),
        "confidence": round(float(result['confidence']), 3),
        "uncertain_flag": int(result['uncertain']),
        "what_to_do": result['action'],
        "when_to_do": result['when']
    })

df_predictions = pd.DataFrame(final_output)

# ── Add supportive messages FIRST ─────────────────────────────────────

def create_supportive_message(row):
    state = row['predicted_state']
    action = row['what_to_do'].replace('_', ' ')
    timing_raw = row['when_to_do'].replace('_', ' ')
    
    timing_map = {
        'now': 'right now',
        'within_15_min': 'within the next 15 minutes',
        'later_today': 'sometime later today',
        'tonight': 'tonight before you sleep',
        'tomorrow_morning': 'tomorrow morning when you’re fresh'
    }
    timing = timing_map.get(timing_raw, timing_raw)
    
    messages = {
        'focused': f"You're already in a focused state — perfect! Go for **{action}** {timing}. Ride that wave.",
        'calm': f"You're in a calm space right now. **{action.capitalize()}** would feel really good {timing}.",
        'restless': f"You seem a bit restless/wired. Let's try **{action}** {timing} to help settle things down.",
        'overwhelmed': f"Things feel heavy — that's okay. **{action.capitalize()}** can give you some relief {timing}.",
        'mixed': f"Signals feel a bit mixed right now. **{action.capitalize()}** should help bring clarity {timing}.",
        'neutral': f"You're in a nice balanced spot. **{action.capitalize()}** could be a smooth next step {timing}."
    }
    
    return messages.get(state, f"Consider **{action}** {timing} — it might feel right.")

df_predictions['supportive_message'] = df_predictions.apply(create_supportive_message, axis=1)

# ── Now select columns (AFTER the message column exists) ──────────────

final_columns = [
    "id",
    "predicted_state",
    "predicted_intensity",
    "confidence",
    "uncertain_flag",
    "what_to_do",
    "when_to_do",
    "supportive_message"
]

# Safety: only take columns that actually exist
available_cols = [col for col in final_columns if col in df_predictions.columns]

df_final = df_predictions[available_cols]

df_final.to_csv("predictions.csv", index=False)

print("Final predictions with supportive messages saved → predictions.csv")
print(f"Rows: {len(df_final)} | Columns: {df_final.columns.tolist()}")
print("\nPreview of first 6 rows:\n")
print(df_final.head(6)[["id", "predicted_state", "what_to_do", "when_to_do", "supportive_message"]])


Final predictions with supportive messages saved → predictions.csv
Rows: 120 | Columns: ['id', 'predicted_state', 'predicted_intensity', 'confidence', 'uncertain_flag', 'what_to_do', 'when_to_do', 'supportive_message']

Preview of first 6 rows:

      id predicted_state      what_to_do     when_to_do  \
0  10001         focused       deep_work            now   
1  10002        restless   sound_therapy            now   
2  10003         neutral            rest    later_today   
3  10004         focused  light_planning  within_15_min   
4  10005            calm            rest    later_today   
5  10006           mixed           pause            now   

                                  supportive_message  
0  You're already in a focused state — perfect! G...  
1  You seem a bit restless/wired. Let's try **sou...  
2  You're in a nice balanced spot. **Rest** could...  
3  You're already in a focused state — perfect! G...  
4  You're in a calm space right now. **Rest** wou...  
5  Signals